# 🤖 Conversational Coaching Assistant — Full PoC
**Stack:** OpenAI GPT-4 · LangChain · FAISS · FastAPI · Intelligent Scoring · User Profiling

This notebook walks through all 6 phases end-to-end:
1. Environment setup
2. OpenAI baseline chat
3. RAG pipeline (FAISS + embeddings)
4. Multi-LLM router
5. Intelligent scorer
6. User profiling + personalization
7. FastAPI app (runnable from notebook)
8. Interactive demo UI (ipywidgets)

---
## Phase 0 — Install dependencies

In [8]:
# Run once — restart kernel after this cell
!pip install openai langchain langchain-openai langchain-community faiss-cpu \
             fastapi uvicorn pydantic python-dotenv ipywidgets tiktoken \
             pypdf httpx nest-asyncio -q

---
## Phase 1 — Configuration & OpenAI baseline

In [9]:
import os
import json
import time
from datetime import datetime
from typing import Optional

# ── Set your key here (or use a .env file) ──────────────────────────────────
OPENAI_API_KEY = "sk-proj-80qIfA7ZyPOGE8zs...."   
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Active model — change to swap providers in Phase 3
ACTIVE_MODEL = "openai"          # "openai" | "ollama" | "mistral"
OPENAI_CHAT_MODEL = "gpt-4o-mini" # cheaper than gpt-4, same API
EMBED_MODEL = "text-embedding-3-small"

print("✅ Config ready")

✅ Config ready


In [10]:
from openai import OpenAI

client = OpenAI()

def chat_openai(messages: list[dict], model: str = OPENAI_CHAT_MODEL) -> str:
    """Single call to OpenAI chat completion. Returns assistant text."""
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.7,
    )
    return response.choices[0].message.content

# Quick smoke test
test = chat_openai([{"role": "user", "content": "Say 'coaching assistant ready' in 5 words."}])
print("Model says:", test)

Model says: Coaching assistant is ready now.


---
## Phase 2 — Knowledge base & RAG pipeline

In [11]:
# ── Sample coaching knowledge base ──────────────────────────────────────────
COACHING_DOCUMENTS = [    # ← added [ here and renamed to COACHING_DOCUMENTS

    # ── FRAMEWORKS ────────────────────────────────────────────────────────────
    {
        "id": "doc_001",
        "category": "framework",
        "title": "The GROW Model",
        "content": """The GROW model is the most widely used coaching framework in the world.
It stands for: Goal (what do you want to achieve?), Reality (where are you now?),
Options (what could you do?), and Will (what will you actually commit to doing, and by when?).
Good coaches spend equal time on all four stages. Most managers skip directly to Options,
which leads to advice-giving rather than coaching. Always start by clarifying the Goal."""
    },
    {
        "id": "doc_002",
        "category": "framework",
        "title": "The SBI Feedback Model",
        "content": """SBI stands for Situation, Behaviour, Impact — the gold standard for giving feedback.
Situation: describe the specific context (when and where). Behaviour: describe the observable
action, not the personality. Impact: explain the effect it had on the team or work.
Example: 'In yesterday's team meeting (S), you interrupted Sarah three times (B),
which made her stop contributing for the rest of the session (I).'
Avoid saying 'you always' or 'you never' — stick to the specific event."""
    },
    {
        "id": "doc_003",
        "category": "framework",
        "title": "The Coaching vs Managing Distinction",
        "content": """Coaching, mentoring, and managing are three different modes leaders use.
Managing is directive: you set tasks, monitor output, and evaluate results.
Mentoring is advisory: you share your own experience and suggest what worked for you.
Coaching is non-directive: you ask questions to help the person find their own answers.
The best managers switch between all three depending on the situation.
Use coaching when the person has the capability but needs confidence or clarity.
Use managing when there is a clear performance or compliance issue."""
    },
    {
        "id": "doc_004",
        "category": "framework",
        "title": "Active Listening in Coaching",
        "content": """Active listening is the foundation of effective coaching.
It means giving full attention, not planning your response while the other person speaks.
Key techniques: reflect back what you heard ('So what I'm hearing is...'),
ask clarifying questions ('Can you tell me more about that?'),
use silence — pause for 3-5 seconds after someone speaks before responding.
Avoid: finishing sentences, jumping to solutions, checking your phone.
Studies show managers typically listen for only 17 seconds before interrupting."""
    },
    {
        "id": "doc_005",
        "category": "framework",
        "title": "Psychological Safety",
        "content": """Psychological safety, defined by Amy Edmondson of Harvard, is the belief that
one can speak up, take risks, and make mistakes without fear of punishment or humiliation.
Teams with high psychological safety are more innovative, learn faster, and perform better.
Managers build it by: modelling vulnerability (admitting their own mistakes),
rewarding people who raise problems early, never punishing honest mistakes,
and separating the person from the error during feedback conversations."""
    },

    # ── 1-ON-1 MEETINGS ───────────────────────────────────────────────────────
    {
        "id": "doc_006",
        "category": "one_on_one",
        "title": "Running Effective 1-on-1 Meetings",
        "content": """One-on-one meetings are the single most important coaching tool for managers.
Best practices: hold them weekly for 30-45 minutes, never cancel them,
let the team member set the agenda, start by asking 'What's most important for you today?'
Spend 70% of the time listening. Follow up on commitments made in the previous session.
Avoid status updates — that's what team meetings are for.
The 1-on-1 is for the person's development, growth, and wellbeing."""
    },
    {
        "id": "doc_007",
        "category": "one_on_one",
        "title": "Questions to Ask in a 1-on-1",
        "content": """Powerful 1-on-1 questions include:
'What's going well that we should do more of?'
'What's one thing I could do differently to better support you?'
'What's the biggest obstacle you're facing right now?'
'Where do you want to be in 12 months, and how can I help?'
'On a scale of 1-10, how energised do you feel about your work right now?'
Avoid yes/no questions. Always follow up with 'Tell me more about that.'
Write down the answers — it shows you take the conversation seriously."""
    },
    {
        "id": "doc_008",
        "category": "one_on_one",
        "title": "Handling Difficult 1-on-1 Conversations",
        "content": """When a 1-on-1 touches on difficult topics (performance issues, personal problems,
conflict with colleagues), use the COIN model: Context, Observation, Impact, Next steps.
Start neutral: 'I wanted to check in about something I've noticed.'
Never ambush — give a heads up before the meeting if the topic is sensitive.
If the person becomes emotional, pause and acknowledge: 'I can see this is difficult.
Take your time.' Never rush through a hard conversation to get back to your schedule."""
    },

    # ── PERFORMANCE & MOTIVATION ───────────────────────────────────────────────
    {
        "id": "doc_009",
        "category": "performance",
        "title": "Dealing with Underperformance",
        "content": """Underperformance has two root causes: capability (they can't do it) or
motivation (they won't do it). Diagnose before acting.
Ask: 'Have they done this well before?' If yes → motivation issue.
If no → capability issue requiring training or clearer expectations.
For motivation issues, explore: Do they understand why the work matters?
Is something blocking them outside work? Have expectations been clearly stated?
Document conversations and agreed actions. Never surprise someone with a formal review."""
    },
    {
        "id": "doc_010",
        "category": "performance",
        "title": "Motivating High Performers",
        "content": """High performers have different needs than average performers.
They are often motivated by: autonomy (freedom to decide how to do their work),
mastery (opportunities to grow and learn), and purpose (doing work that matters).
Warning signs a high performer is disengaging: they stop asking questions,
they contribute less in meetings, they stop pushing back on decisions.
Interventions: give them stretch assignments, involve them in strategic decisions,
connect them with senior leaders, and ask them what they want to learn next."""
    },
    {
        "id": "doc_011",
        "category": "performance",
        "title": "Setting Clear Expectations",
        "content": """Most performance problems come from unclear expectations, not lack of ability.
Use the SMART framework: Specific (what exactly should be done?),
Measurable (how will we know it's done?), Achievable (is it realistic?),
Relevant (does it connect to team goals?), Time-bound (by when?).
After explaining an expectation, always ask: 'Can you tell me back what you understand
the goal to be?' This catches misunderstandings early without being condescending."""
    },
    {
        "id": "doc_012",
        "category": "performance",
        "title": "Giving Positive Recognition",
        "content": """Recognition is one of the cheapest and most underused management tools.
Research shows employees need 5 positive interactions for every 1 negative one
to maintain engagement (the Losada ratio).
Effective recognition is: specific ('The way you handled that client complaint was excellent
because you stayed calm and proposed a solution immediately'), timely (within 24-48 hours),
and delivered in the person's preferred way (some people hate public praise).
Never give recognition that sounds like a preamble to criticism ('Great job, but...')."""
    },

    # ── TEAM DYNAMICS ─────────────────────────────────────────────────────────
    {
        "id": "doc_013",
        "category": "team",
        "title": "Managing Team Conflict",
        "content": """Not all conflict is bad. Productive conflict around ideas drives innovation.
Destructive conflict — personal attacks, passive aggression, blame — damages performance.
When conflict arises: first meet each person separately to hear their perspective.
Then bring them together and set ground rules: speak about your own experience,
not the other person's intent. Focus on the issue, not the person.
Your role as coach is facilitator, not judge. Don't take sides.
End with agreed actions and a follow-up check-in."""
    },
    {
        "id": "doc_014",
        "category": "team",
        "title": "Building Team Autonomy",
        "content": """Autonomous teams make better decisions faster and are more engaged.
To build autonomy: start by delegating small decisions, then increase scope as trust grows.
Use the Delegation Poker framework — for each type of decision, agree who decides:
Manager decides, Manager decides with input, Team decides with manager input, Team decides.
The most common mistake: managers say they want autonomy but override decisions anyway.
If you delegate a decision, accept the outcome even if you would have done it differently."""
    },
    {
        "id": "doc_015",
        "category": "team",
        "title": "Remote and Hybrid Team Coaching",
        "content": """Remote teams need more intentional coaching than co-located ones.
Best practices: increase 1-on-1 frequency (weekly, not monthly), use video for sensitive
conversations (never text or email), create virtual 'watercooler' moments for connection.
Watch for isolation signals: camera always off, slow reply times, short answers.
Asynchronous communication removes tone — be extra explicit about intent and feeling.
Celebrate wins publicly in team channels. Deliver feedback privately and synchronously."""
    },

    # ── COACHING CONVERSATIONS ─────────────────────────────────────────────────
    {
        "id": "doc_016",
        "category": "conversation",
        "title": "Asking Powerful Questions",
        "content": """Powerful questions open up thinking rather than close it down.
They are usually open-ended, short, and focused on the future.
Examples: 'What would you do if you knew you couldn't fail?'
'What's the real challenge here for you?'
'If a colleague you respect faced this situation, what would you advise them?'
'What haven't you tried yet?'
Avoid 'why' questions — they can feel accusatory. Replace with 'what led you to...' or
'help me understand your thinking on...'."""
    },
    {
        "id": "doc_017",
        "category": "conversation",
        "title": "Handling Resistance in Coaching",
        "content": """Resistance (eye-rolling, short answers, deflection) is information, not obstruction.
It usually means: the person doesn't trust the process, feels judged, or is not ready.
Don't push harder — that increases resistance. Instead, name it gently:
'I notice you seem hesitant. What's your concern about this conversation?'
Sometimes resistance signals the wrong goal has been set.
Go back to basics: 'What would make this conversation useful for you today?'
Resistance often dissolves when the person feels genuinely heard."""
    },
    {
        "id": "doc_018",
        "category": "conversation",
        "title": "Closing a Coaching Conversation",
        "content": """A good coaching conversation ends with clarity, not just a warm feeling.
Always close with: a specific action ('What will you do, and by when?'),
accountability ('How will I know you've done it?'),
and reflection ('What's the one thing you're taking away from our conversation today?').
Write down the commitment — both of you should have a copy.
At the next session, start by reviewing commitments before setting new ones.
Accountability without judgment is the hallmark of a great coaching relationship."""
    },

    # ── LEADERSHIP & DEVELOPMENT ───────────────────────────────────────────────
    {
        "id": "doc_019",
        "category": "leadership",
        "title": "Developing Future Leaders",
        "content": """Identifying and developing future leaders is a core management responsibility.
Look for: people who influence others without authority, who ask strategic questions,
who take initiative without being asked, and who develop others around them.
Development tactics: give them projects with high visibility and real stakes,
involve them in decisions above their level, connect them with senior mentors,
and give them honest feedback about their leadership gaps — not just their technical ones."""
    },
    {
        "id": "doc_020",
        "category": "leadership",
        "title": "First-Time Manager Coaching",
        "content": """The transition from individual contributor to manager is one of the hardest in a career.
Common mistakes first-time managers make: trying to do everything themselves,
avoiding difficult conversations, managing their former peers the same way as before.
Key shifts needed: from 'doing' to 'enabling others to do',
from 'being the expert' to 'asking the right questions',
from 'personal output' to 'team output'.
Coach them to invest in relationships before they need them."""
    },
    {
        "id": "doc_021",
        "category": "leadership",
        "title": "Coaching Across Cultures",
        "content": """Coaching approaches that work in one culture may fail in another.
In high-context cultures (Japan, China, Arab countries), direct feedback can cause shame.
In low-power-distance cultures (Netherlands, Scandinavia), employees expect to challenge managers.
In high-power-distance cultures (France, Mexico, India), the manager's word is rarely questioned.
Adapt your style: in high-context cultures, use indirect questions and check in more frequently.
Always ask: 'How do people in your culture typically give and receive feedback?' before starting."""
    },

    # ── WELLBEING & BURNOUT ────────────────────────────────────────────────────
    {
        "id": "doc_022",
        "category": "wellbeing",
        "title": "Recognising Burnout in Your Team",
        "content": """Burnout is not just tiredness — it is chronic workplace stress that has not been managed.
The three dimensions (Maslach): exhaustion (depleted energy), cynicism (detachment from work),
and reduced efficacy (feeling ineffective).
Early warning signs: arriving late or leaving very early, missing deadlines they normally meet,
shorter responses in messages, withdrawing from team social events.
As a manager: ask directly 'How are you really doing?' and listen.
Never minimise ('Everyone is stressed'). Help them reduce load or adjust priorities immediately."""
    },
    {
        "id": "doc_023",
        "category": "wellbeing",
        "title": "Supporting Employees Through Personal Difficulties",
        "content": """When a team member is going through a personal difficulty (bereavement, illness, divorce),
your role is to be a human first, manager second.
Start with empathy: 'I heard what happened and I'm truly sorry. I want you to know I'm here.'
Offer practical help: flexible hours, reduced workload, covering meetings.
Don't ask for details you don't need. Don't say 'Let me know if there's anything I can do'
— that puts the burden back on them. Instead, make a specific offer.
Know your company's EAP (Employee Assistance Programme) and refer to it when appropriate."""
    },

    # ── INCENTEEV-SPECIFIC ─────────────────────────────────────────────────────
    {
        "id": "doc_024",
        "category": "incenteev",
        "title": "Incenteev Coaching Philosophy",
        "content": """Incenteev believes that every manager can become a better coach with the right tools.
Our approach is built on three pillars: frequency (regular short coaching conversations
beat rare long ones), personalisation (coaching must adapt to each individual's goals and style),
and data (measuring coaching quality leads to faster improvement).
We work with managers at P&G, Orange, Société Générale, LG and Groupama to embed
a coaching culture across their organisations — not just training a few top leaders."""
    },
    {
        "id": "doc_025",
        "category": "incenteev",
        "title": "Measuring Coaching Effectiveness",
        "content": """Coaching effectiveness can be measured at four levels (Kirkpatrick model adapted):
Level 1 — Reaction: Did the person find the coaching session useful? (post-session survey)
Level 2 — Learning: Did they gain new insight or perspective? (reflection questions)
Level 3 — Behaviour: Did they apply what they discussed? (follow-up at next session)
Level 4 — Results: Did team performance improve? (KPIs, engagement scores, retention)
Incenteev tracks all four levels to show clients the ROI of coaching at scale."""
    },
]   # ← closing bracket was already there, now it matches

# Extract just the content for FAISS (this is what RAG uses)
COACHING_DOCS = [d["content"] for d in COACHING_DOCUMENTS]

print(f"✅ {len(COACHING_DOCUMENTS)} coaching documents created")
for doc in COACHING_DOCUMENTS:
    print(f"  [{doc['category']:12s}] {doc['id']} — {doc['title']}")

✅ 25 coaching documents created
  [framework   ] doc_001 — The GROW Model
  [framework   ] doc_002 — The SBI Feedback Model
  [framework   ] doc_003 — The Coaching vs Managing Distinction
  [framework   ] doc_004 — Active Listening in Coaching
  [framework   ] doc_005 — Psychological Safety
  [one_on_one  ] doc_006 — Running Effective 1-on-1 Meetings
  [one_on_one  ] doc_007 — Questions to Ask in a 1-on-1
  [one_on_one  ] doc_008 — Handling Difficult 1-on-1 Conversations
  [performance ] doc_009 — Dealing with Underperformance
  [performance ] doc_010 — Motivating High Performers
  [performance ] doc_011 — Setting Clear Expectations
  [performance ] doc_012 — Giving Positive Recognition
  [team        ] doc_013 — Managing Team Conflict
  [team        ] doc_014 — Building Team Autonomy
  [team        ] doc_015 — Remote and Hybrid Team Coaching
  [conversation] doc_016 — Asking Powerful Questions
  [conversation] doc_017 — Handling Resistance in Coaching
  [conversation] doc_018 — Closin

In [12]:
import numpy as np
import faiss

def embed_texts(texts: list[str]) -> np.ndarray:
    """Embed a list of texts using OpenAI embeddings."""
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    vectors = [item.embedding for item in response.data]
    return np.array(vectors, dtype="float32")

def build_faiss_index(docs: list[str]):
    """Build a FAISS index from a list of text documents."""
    print("Embedding documents...")
    vectors = embed_texts(docs)
    dim = vectors.shape[1]
    index = faiss.IndexFlatIP(dim)   # Inner product = cosine on normalised vecs
    faiss.normalize_L2(vectors)
    index.add(vectors)
    print(f"✅ FAISS index built: {index.ntotal} vectors, dim={dim}")
    return index

FAISS_INDEX = build_faiss_index(COACHING_DOCS)

Embedding documents...
✅ FAISS index built: 25 vectors, dim=1536


In [13]:
def retrieve(query: str, k: int = 3) -> list[str]:
    """Return the top-k most relevant coaching documents for a query."""
    q_vec = embed_texts([query])
    faiss.normalize_L2(q_vec)
    distances, indices = FAISS_INDEX.search(q_vec, k)
    return [COACHING_DOCS[i] for i in indices[0] if i < len(COACHING_DOCS)]

# Test retrieval
results = retrieve("How do I give better feedback to my team?")
print("Top retrieved chunk:")
print(results[0])

Top retrieved chunk:
Remote teams need more intentional coaching than co-located ones.
Best practices: increase 1-on-1 frequency (weekly, not monthly), use video for sensitive
conversations (never text or email), create virtual 'watercooler' moments for connection.
Watch for isolation signals: camera always off, slow reply times, short answers.
Asynchronous communication removes tone — be extra explicit about intent and feeling.
Celebrate wins publicly in team channels. Deliver feedback privately and synchronously.


---
## Phase 3 — Multi-LLM router

In [14]:
import httpx

def chat_ollama(messages: list[dict], model: str = "llama3") -> str:
    """Call a local Ollama instance. Requires: `ollama serve` running."""
    payload = {"model": model, "messages": messages, "stream": False}
    r = httpx.post("http://localhost:11434/api/chat", json=payload, timeout=60)
    return r.json()["message"]["content"]


def chat_mistral(messages: list[dict], model: str = "mistral-small-latest") -> str:
    """Call Mistral API. Requires MISTRAL_API_KEY env var."""
    mistral_key = os.getenv("MISTRAL_API_KEY", "")
    headers = {"Authorization": f"Bearer {mistral_key}", "Content-Type": "application/json"}
    payload = {"model": model, "messages": messages}
    r = httpx.post("https://api.mistral.ai/v1/chat/completions", json=payload, headers=headers, timeout=30)
    return r.json()["choices"][0]["message"]["content"]


def llm_router(messages: list[dict], provider: str = ACTIVE_MODEL) -> str:
    """
    Route to the right LLM based on ACTIVE_MODEL.
    Switch provider by changing ACTIVE_MODEL at the top of Phase 1.
    """
    if provider == "openai":
        return chat_openai(messages)
    elif provider == "ollama":
        return chat_ollama(messages)
    elif provider == "mistral":
        return chat_mistral(messages)
    else:
        raise ValueError(f"Unknown provider: {provider}")

print(f"✅ LLM router ready — active provider: {ACTIVE_MODEL}")

✅ LLM router ready — active provider: openai


---
## Phase 4 — Intelligent scorer

In [15]:
from pydantic import BaseModel

class CoachingScore(BaseModel):
    relevance: int           # 1-5: Does the response address the question?
    specificity: int         # 1-5: Is advice concrete, not generic?
    actionability: int       # 1-5: Can the user act on this immediately?
    empathy: int             # 1-5: Is the tone supportive and human?
    overall: int             # 1-5: Overall coaching quality
    feedback: str            # One sentence explaining the main strength/gap


SCORER_SYSTEM = """You are an expert coaching quality evaluator.
Given a user question and an AI coaching response, score the response on 5 dimensions (1=poor, 5=excellent):
- relevance: Does it address what the user asked?
- specificity: Is the advice concrete rather than generic?
- actionability: Can the user act on it right now?
- empathy: Is the tone warm and supportive?
- overall: Overall coaching quality.
Also write one short feedback sentence.
Respond ONLY with valid JSON matching this schema exactly:
{"relevance": int, "specificity": int, "actionability": int, "empathy": int, "overall": int, "feedback": str}"""


def score_response(user_question: str, assistant_response: str) -> CoachingScore:
    """Use GPT-4 as a judge to score a coaching response."""
    prompt = f"User question: {user_question}\n\nCoaching response: {assistant_response}"
    messages = [
        {"role": "system", "content": SCORER_SYSTEM},
        {"role": "user", "content": prompt},
    ]
    raw = chat_openai(messages)
    data = json.loads(raw)
    return CoachingScore(**data)


# Test the scorer
test_q = "How can I give better feedback?"
test_a = "Give feedback clearly and respectfully."
score = score_response(test_q, test_a)
print("Score:", score.model_dump())

Score: {'relevance': 5, 'specificity': 2, 'actionability': 4, 'empathy': 3, 'overall': 3, 'feedback': 'The response is relevant but could benefit from more specific examples and a warmer tone.'}


---
## Phase 5 — User profiles & personalization

In [16]:
USER_PROFILES = [

    # ── P&G ───────────────────────────────────────────────────────────────────
    {
        "user_id": "pg_001",
        "name": "Alice Martin",
        "role": "Sales Manager",
        "company": "Procter & Gamble",
        "country": "France",
        "experience_years": 3,
        "team_size": 8,
        "coaching_style": "non-directive",
        "goals": ["Improve 1-on-1 quality", "Give better feedback", "Handle underperformers"],
        "challenges": ["Team member missing deadlines", "Difficulty delegating"],
        "session_count": 12,
        "language": "French",
        "last_session": "2026-04-10"
    },
    {
        "user_id": "pg_002",
        "name": "James Okonkwo",
        "role": "Regional Director",
        "company": "Procter & Gamble",
        "country": "Nigeria",
        "experience_years": 11,
        "team_size": 24,
        "coaching_style": "directive",
        "goals": ["Develop future leaders", "Reduce team turnover", "Scale coaching culture"],
        "challenges": ["High performer disengagement", "Cross-cultural management"],
        "session_count": 5,
        "language": "English",
        "last_session": "2026-04-18"
    },
    {
        "user_id": "pg_003",
        "name": "Sophie Beaumont",
        "role": "HR Business Partner",
        "company": "Procter & Gamble",
        "country": "Belgium",
        "experience_years": 7,
        "team_size": 5,
        "coaching_style": "balanced",
        "goals": ["Support manager development", "Improve psychological safety"],
        "challenges": ["Getting managers to embrace coaching", "Measuring impact"],
        "session_count": 8,
        "language": "French",
        "last_session": "2026-04-15"
    },

    # ── ORANGE ────────────────────────────────────────────────────────────────
    {
        "user_id": "or_001",
        "name": "Mohammed Al-Rashid",
        "role": "Engineering Lead",
        "company": "Orange",
        "country": "Jordan",
        "experience_years": 6,
        "team_size": 12,
        "coaching_style": "directive",
        "goals": ["Build team autonomy", "Manage remote engineers", "Reduce micromanagement"],
        "challenges": ["Team relies on him for every decision", "Struggles to give negative feedback"],
        "session_count": 15,
        "language": "English",
        "last_session": "2026-04-22"
    },
    {
        "user_id": "or_002",
        "name": "Camille Dupont",
        "role": "Customer Experience Manager",
        "company": "Orange",
        "country": "France",
        "experience_years": 4,
        "team_size": 15,
        "coaching_style": "balanced",
        "goals": ["Reduce team burnout", "Improve NPS scores", "Better 1-on-1 conversations"],
        "challenges": ["Two team members in conflict", "High workload pressure"],
        "session_count": 9,
        "language": "French",
        "last_session": "2026-04-20"
    },
    {
        "user_id": "or_003",
        "name": "Lena Schneider",
        "role": "Product Manager",
        "company": "Orange",
        "country": "Germany",
        "experience_years": 2,
        "team_size": 6,
        "coaching_style": "non-directive",
        "goals": ["Grow as a first-time manager", "Build team trust"],
        "challenges": ["Managing former peers", "Imposter syndrome"],
        "session_count": 20,
        "language": "English",
        "last_session": "2026-04-23"
    },

    # ── SOCIÉTÉ GÉNÉRALE ──────────────────────────────────────────────────────
    {
        "user_id": "sg_001",
        "name": "Marie Fontaine",
        "role": "Risk Management Director",
        "company": "Société Générale",
        "country": "France",
        "experience_years": 14,
        "team_size": 30,
        "coaching_style": "directive",
        "goals": ["Develop next generation leaders", "Improve team engagement scores"],
        "challenges": ["Retention of top talent", "Managing a geographically dispersed team"],
        "session_count": 3,
        "language": "French",
        "last_session": "2026-04-05"
    },
    {
        "user_id": "sg_002",
        "name": "Thomas Bergmann",
        "role": "IT Project Manager",
        "company": "Société Générale",
        "country": "Germany",
        "experience_years": 8,
        "team_size": 10,
        "coaching_style": "balanced",
        "goals": ["Improve project delivery", "Better stakeholder communication"],
        "challenges": ["Team misses deadlines", "Difficult senior stakeholder"],
        "session_count": 11,
        "language": "English",
        "last_session": "2026-04-17"
    },

    # ── LG ────────────────────────────────────────────────────────────────────
    {
        "user_id": "lg_001",
        "name": "Ji-ho Kim",
        "role": "R&D Team Lead",
        "company": "LG",
        "country": "South Korea",
        "experience_years": 9,
        "team_size": 18,
        "coaching_style": "directive",
        "goals": ["Encourage more initiative from team", "Reduce hierarchy in communication"],
        "challenges": ["Team waits for instructions", "Cultural resistance to feedback"],
        "session_count": 7,
        "language": "English",
        "last_session": "2026-04-12"
    },
    {
        "user_id": "lg_002",
        "name": "Yuna Park",
        "role": "Marketing Manager",
        "company": "LG",
        "country": "South Korea",
        "experience_years": 5,
        "team_size": 9,
        "coaching_style": "non-directive",
        "goals": ["Empower team creativity", "Improve cross-functional collaboration"],
        "challenges": ["Silos between departments", "Team lacks confidence to propose ideas"],
        "session_count": 13,
        "language": "English",
        "last_session": "2026-04-19"
    },

    # ── GROUPAMA ──────────────────────────────────────────────────────────────
    {
        "user_id": "gr_001",
        "name": "Pierre Leroy",
        "role": "Branch Manager",
        "company": "Groupama",
        "country": "France",
        "experience_years": 16,
        "team_size": 20,
        "coaching_style": "directive",
        "goals": ["Modernise management style", "Attract younger talent"],
        "challenges": ["Old habits from command-and-control culture", "Digital transformation resistance"],
        "session_count": 4,
        "language": "French",
        "last_session": "2026-04-08"
    },
    {
        "user_id": "gr_002",
        "name": "Isabelle Moreau",
        "role": "Claims Team Supervisor",
        "company": "Groupama",
        "country": "France",
        "experience_years": 3,
        "team_size": 7,
        "coaching_style": "balanced",
        "goals": ["Reduce team stress", "Improve quality of customer calls"],
        "challenges": ["High call volume pressure", "One team member with attitude problem"],
        "session_count": 17,
        "language": "French",
        "last_session": "2026-04-21"
    },

    # ── MIX OF OTHER COMPANIES ────────────────────────────────────────────────
    {
        "user_id": "ot_001",
        "name": "Carlos Rivera",
        "role": "Operations Manager",
        "company": "Renault",
        "country": "Spain",
        "experience_years": 10,
        "team_size": 35,
        "coaching_style": "directive",
        "goals": ["Improve factory floor communication", "Reduce absenteeism"],
        "challenges": ["Union tensions", "Language barriers in multicultural team"],
        "session_count": 6,
        "language": "English",
        "last_session": "2026-04-14"
    },
    {
        "user_id": "ot_002",
        "name": "Amara Diallo",
        "role": "Customer Success Lead",
        "company": "Thales",
        "country": "Senegal",
        "experience_years": 4,
        "team_size": 11,
        "coaching_style": "non-directive",
        "goals": ["Build client relationship skills in team", "Improve retention metrics"],
        "challenges": ["Junior team needing a lot of support", "High expectations from leadership"],
        "session_count": 10,
        "language": "French",
        "last_session": "2026-04-16"
    },
    {
        "user_id": "ot_003",
        "name": "Elena Petrov",
        "role": "Data Science Manager",
        "company": "BNP Paribas",
        "country": "Romania",
        "experience_years": 5,
        "team_size": 8,
        "coaching_style": "non-directive",
        "goals": ["Support career growth of data scientists", "Improve team collaboration"],
        "challenges": ["High performers want to leave for startups", "Slow promotion processes"],
        "session_count": 14,
        "language": "English",
        "last_session": "2026-04-24"
    },
    {
        "user_id": "ot_004",
        "name": "David Osei",
        "role": "Supply Chain Manager",
        "company": "Michelin",
        "country": "Ghana",
        "experience_years": 8,
        "team_size": 16,
        "coaching_style": "balanced",
        "goals": ["Develop team problem-solving skills", "Improve KPI ownership"],
        "challenges": ["Team blames external factors for missed targets", "Lack of data literacy"],
        "session_count": 8,
        "language": "English",
        "last_session": "2026-04-11"
    },
    {
        "user_id": "ot_005",
        "name": "Nathalie Clement",
        "role": "L&D Manager",
        "company": "Air France",
        "country": "France",
        "experience_years": 12,
        "team_size": 4,
        "coaching_style": "balanced",
        "goals": ["Roll out coaching culture company-wide", "Train internal coaches"],
        "challenges": ["Budget constraints", "Getting buy-in from top management"],
        "session_count": 2,
        "language": "French",
        "last_session": "2026-04-03"
    },
    {
        "user_id": "ot_006",
        "name": "Rafael Torres",
        "role": "Sales Team Lead",
        "company": "Schneider Electric",
        "country": "Mexico",
        "experience_years": 6,
        "team_size": 13,
        "coaching_style": "directive",
        "goals": ["Hit Q2 targets", "Improve cold call conversion", "Reduce team churn"],
        "challenges": ["Two reps consistently underperforming", "Team morale low after restructuring"],
        "session_count": 9,
        "language": "English",
        "last_session": "2026-04-20"
    },
    {
        "user_id": "ot_007",
        "name": "Priya Nair",
        "role": "Engineering Manager",
        "company": "Capgemini",
        "country": "India",
        "experience_years": 7,
        "team_size": 22,
        "coaching_style": "non-directive",
        "goals": ["Improve psychological safety", "Reduce code review conflicts"],
        "challenges": ["Senior engineers dismissing junior ideas", "Hybrid team coordination"],
        "session_count": 16,
        "language": "English",
        "last_session": "2026-04-23"
    },
    {
        "user_id": "ot_008",
        "name": "Florian Weber",
        "role": "COO",
        "company": "Dassault Systèmes",
        "country": "Germany",
        "experience_years": 20,
        "team_size": 120,
        "coaching_style": "balanced",
        "goals": ["Create coaching culture at exec level", "Succession planning"],
        "challenges": ["Board pressure on short-term results", "Losing senior talent to competitors"],
        "session_count": 1,
        "language": "English",
        "last_session": "2026-04-01"
    },
    {
        "user_id": "ot_009",
        "name": "Aiko Tanaka",
        "role": "Quality Assurance Lead",
        "company": "Sanofi",
        "country": "Japan",
        "experience_years": 9,
        "team_size": 14,
        "coaching_style": "non-directive",
        "goals": ["Encourage team to flag issues early", "Build psychological safety"],
        "challenges": ["Cultural reluctance to report problems", "Perfectionism causing delays"],
        "session_count": 11,
        "language": "English",
        "last_session": "2026-04-18"
    },
    {
        "user_id": "ot_010",
        "name": "Lucas Bernard",
        "role": "Retail Store Manager",
        "company": "LVMH",
        "country": "France",
        "experience_years": 3,
        "team_size": 10,
        "coaching_style": "balanced",
        "goals": ["Improve client experience scores", "Reduce staff turnover"],
        "challenges": ["Seasonal staff hard to integrate", "High performance pressure from HQ"],
        "session_count": 7,
        "language": "French",
        "last_session": "2026-04-22"
    },
]

print(f"✅ {len(USER_PROFILES)} user profiles created")
companies = {}
for u in USER_PROFILES:
    companies.setdefault(u['company'], 0)
    companies[u['company']] += 1
for company, count in sorted(companies.items()):
    print(f"  {company}: {count} user(s)")

✅ 22 user profiles created
  Air France: 1 user(s)
  BNP Paribas: 1 user(s)
  Capgemini: 1 user(s)
  Dassault Systèmes: 1 user(s)
  Groupama: 2 user(s)
  LG: 2 user(s)
  LVMH: 1 user(s)
  Michelin: 1 user(s)
  Orange: 3 user(s)
  Procter & Gamble: 3 user(s)
  Renault: 1 user(s)
  Sanofi: 1 user(s)
  Schneider Electric: 1 user(s)
  Société Générale: 2 user(s)
  Thales: 1 user(s)


In [17]:
from dataclasses import dataclass, field, asdict
import json

@dataclass
class UserProfile:
    user_id: str
    name: str
    role: str                          # e.g. "Sales manager"
    company: str
    goals: list[str] = field(default_factory=list)   # e.g. ["Improve 1-on-1s"]
    coaching_style: str = "balanced"   # "directive" | "non-directive" | "balanced"
    topics_discussed: list[str] = field(default_factory=list)
    session_count: int = 0

    def system_prompt(self) -> str:
        """Build a personalised system prompt from this profile."""
        goals_str = ", ".join(self.goals) if self.goals else "not yet set"
        style_instructions = {
            "directive":     "Give clear, direct advice and action steps.",
            "non-directive": "Ask questions to guide the user to their own answers.",
            "balanced":      "Mix questions with practical advice. Be warm but concrete.",
        }[self.coaching_style]
        return f"""You are an expert coaching assistant for managers.
You are speaking with {self.name}, a {self.role} at {self.company}.
Their current coaching goals are: {goals_str}.
Coaching style preference: {style_instructions}
Session number: {self.session_count + 1}.
Always be empathetic, specific and practical. Keep responses concise (under 200 words).
Use the retrieved context below to ground your answer when relevant."""


# In-memory profile store (replace with SQLite for persistence)
PROFILES: dict[str, UserProfile] = {}

def get_or_create_profile(user_id: str, **kwargs) -> UserProfile:
    if user_id not in PROFILES:
        PROFILES[user_id] = UserProfile(user_id=user_id, **kwargs)
    return PROFILES[user_id]


# Load all 25 synthetic profiles from file
with open("profiles/users.json") as f:
    users_data = json.load(f)

# Load profiles directly from USER_PROFILES list defined above (no JSON file needed)
for u in USER_PROFILES:
    get_or_create_profile(
        user_id=u["user_id"],
        name=u["name"],
        role=u["role"],
        company=u["company"],
        goals=u["goals"],
        coaching_style=u["coaching_style"]
    )

print(f"✅ {len(USER_PROFILES)} profiles loaded")

# Preview one profile
alice = PROFILES["pg_001"]  # Alice Martin from P&G
print("Alice system prompt preview:")
print(alice.system_prompt()[:300], "...")

print("✅ Profiles ready")
print("Alice system prompt preview:")
print(alice.system_prompt()[:300], "...")

✅ 22 profiles loaded
Alice system prompt preview:
You are an expert coaching assistant for managers.
You are speaking with Alice Martin, a Sales Manager at Procter & Gamble.
Their current coaching goals are: Improve 1-on-1 quality, Give better feedback, Handle underperformers.
Coaching style preference: Ask questions to guide the user to their own  ...
✅ Profiles ready
Alice system prompt preview:
You are an expert coaching assistant for managers.
You are speaking with Alice Martin, a Sales Manager at Procter & Gamble.
Their current coaching goals are: Improve 1-on-1 quality, Give better feedback, Handle underperformers.
Coaching style preference: Ask questions to guide the user to their own  ...


---
## Phase 6 — Core coaching engine (RAG + Profile + Scorer)

In [18]:
@dataclass
class CoachingSession:
    """Manages conversation history for one user session."""
    profile: UserProfile
    history: list[dict] = field(default_factory=list)

    def chat(self, user_message: str, score: bool = False) -> dict:
        """
        Full pipeline:
        1. Retrieve relevant context from FAISS
        2. Build prompt (system + context + history + user message)
        3. Call LLM router
        4. Optionally score the response
        5. Update history and profile stats
        """
        # Step 1 — RAG retrieval
        context_chunks = retrieve(user_message, k=2)
        context_str = "\n---\n".join(context_chunks)

        # Step 2 — Build system prompt with context
        system = self.profile.system_prompt()
        system += f"\n\n[Relevant coaching knowledge]\n{context_str}"

        # Step 3 — Assemble messages
        messages = [{"role": "system", "content": system}]
        messages += self.history
        messages.append({"role": "user", "content": user_message})

        # Step 4 — Call LLM
        t0 = time.time()
        response_text = llm_router(messages)
        latency = round(time.time() - t0, 2)

        # Step 5 — Update history
        self.history.append({"role": "user", "content": user_message})
        self.history.append({"role": "assistant", "content": response_text})
        self.profile.session_count += 1

        result = {
            "response": response_text,
            "context_used": context_chunks,
            "latency_s": latency,
            "model": ACTIVE_MODEL,
        }

        # Step 6 — Optional scoring
        if score:
            result["score"] = score_response(user_message, response_text).model_dump()

        return result

print("✅ CoachingSession engine ready")

✅ CoachingSession engine ready


In [19]:
# ── Test with Alice ──────────────────────────────────────────────────────────
session_alice = CoachingSession(profile=alice)

result = session_alice.chat(
    "My team member keeps missing deadlines. How should I approach the next 1-on-1?",
    score=True
)

print("🗣️  Response:")
print(result["response"])
print(f"\n⏱️  Latency: {result['latency_s']}s | Model: {result['model']}")
print("\n📊 Score:")
for k, v in result["score"].items():
    print(f"  {k}: {v}")

🗣️  Response:
It’s great you’re looking to address this in your next 1-on-1. To start, consider giving your team member a heads up about discussing their deadlines. This prepares them for the conversation and avoids any surprises. 

During the meeting, use the COIN model. Begin with the Context: “I wanted to check in about something I’ve noticed regarding deadlines.” 

Next, share your Observations: “I’ve seen that some tasks have been missed recently.” 

Then, discuss the Impact: “This affects not only our team’s productivity but also your overall progress.” 

Finally, outline the Next Steps: “How can we work together to ensure deadlines are met moving forward? What support do you need from me?”

Remember to listen actively and be empathetic. If the conversation becomes emotional, pause and acknowledge their feelings. This approach can help your team member feel supported while addressing the issue effectively. How does this plan resonate with you?

⏱️  Latency: 5.85s | Model: openai


In [20]:
# ── Test with two different profiles from synthetic data ─────────────────────

# Alice Martin — P&G — non-directive style
alice = PROFILES["pg_001"]
session_alice = CoachingSession(profile=alice)

result_alice = session_alice.chat(
    "My team member keeps missing deadlines. How should I approach the next 1-on-1?"
)
print("🗣️  Alice's coaching (non-directive style):")
print(result_alice["response"])

print("\n" + "="*60 + "\n")

# Mohammed — Orange — directive style
bob = PROFILES["or_001"]
session_bob = CoachingSession(profile=bob)

result_bob = session_bob.chat(
    "My team member keeps missing deadlines. How should I approach the next 1-on-1?"
)
print("🗣️  Mohammed's coaching (directive style):")
print(result_bob["response"])

print("\n💡 Notice how the tone differs between the two styles!")

🗣️  Alice's coaching (non-directive style):
It’s great that you want to address this in your next 1-on-1. To handle the situation effectively, consider using the COIN model. Start by giving your team member a heads up about discussing performance in advance, ensuring they feel prepared.

In the meeting, begin with context: "I wanted to check in about something I've noticed regarding deadlines." Then, share your observation: "I've seen that a few deadlines have been missed recently." 

Next, communicate the impact: "This affects not only our team's performance but also your own growth and development." 

Finally, transition to next steps: "What do you think might be causing these delays? How can I support you in meeting your deadlines moving forward?"

Remember to listen actively and allow space for them to express their thoughts and feelings. Acknowledging any emotions they display will help create a safe space for this important conversation. How do you feel about this approach?


🗣️ 

---
## Phase 7 — FastAPI app (run from notebook)

In [21]:
import nest_asyncio
import uvicorn
import threading
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel as PydanticModel

nest_asyncio.apply()  # allows uvicorn inside Jupyter event loop

app = FastAPI(title="Coaching Assistant API", version="1.0")

# In-memory sessions store
SESSIONS: dict[str, CoachingSession] = {}


class ChatRequest(PydanticModel):
    user_id: str
    message: str
    score: bool = False


class ProfileRequest(PydanticModel):
    user_id: str
    name: str
    role: str
    company: str
    goals: list[str] = []
    coaching_style: str = "balanced"


@app.get("/")
def root():
    return {"status": "ok", "model": ACTIVE_MODEL}


@app.post("/profile")
def create_profile(req: ProfileRequest):
    """Create or update a user profile."""
    profile = get_or_create_profile(**req.model_dump())
    SESSIONS[req.user_id] = CoachingSession(profile=profile)
    return {"status": "created", "user_id": req.user_id}


@app.post("/chat")
def chat_endpoint(req: ChatRequest):
    """Send a message and get a coaching response."""
    if req.user_id not in SESSIONS:
        # Auto-create a default profile if none exists
        profile = get_or_create_profile(
            user_id=req.user_id,
            name="User", role="Manager", company="Unknown"
        )
        SESSIONS[req.user_id] = CoachingSession(profile=profile)
    session = SESSIONS[req.user_id]
    result = session.chat(req.message, score=req.score)
    return result


@app.get("/profile/{user_id}")
def get_profile(user_id: str):
    """Retrieve a user profile."""
    if user_id not in PROFILES:
        raise HTTPException(status_code=404, detail="Profile not found")
    return asdict(PROFILES[user_id])


@app.get("/history/{user_id}")
def get_history(user_id: str):
    """Retrieve conversation history for a user."""
    if user_id not in SESSIONS:
        raise HTTPException(status_code=404, detail="Session not found")
    return {"history": SESSIONS[user_id].history}


# Start the server in a background thread
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(1.5)
print("✅ FastAPI running at http://127.0.0.1:8000")
print("📖 Docs at http://127.0.0.1:8000/docs")

✅ FastAPI running at http://127.0.0.1:8000
📖 Docs at http://127.0.0.1:8000/docs


In [22]:
# ── Test the API with httpx ──────────────────────────────────────────────────
import httpx

BASE = "http://127.0.0.1:8000"

# Create a profile
r = httpx.post(f"{BASE}/profile", json={
    "user_id": "demo",
    "name": "Marie",
    "role": "HR Director",
    "company": "Société Générale",
    "goals": ["Develop leaders", "Reduce turnover"],
    "coaching_style": "balanced"
})
print("Profile created:", r.json())

# Send a chat message
r = httpx.post(f"{BASE}/chat", json={
    "user_id": "demo",
    "message": "How can I help a high-performer who seems disengaged lately?",
    "score": True
}, timeout=30)
data = r.json()
print("\n🗣️  Response:", data["response"])
print("\n📊 Score:", data.get("score"))

Profile created: {'status': 'created', 'user_id': 'demo'}

🗣️  Response: It's great that you're looking to support your high performer! Start by having an open and empathetic conversation. Ask them how they are feeling about their work and if there are any specific challenges they’re facing. 

To diagnose the issue, consider these questions:
- Have they performed well in this area before? If yes, it could be a motivation issue.
- Do they understand the purpose behind their tasks? Clarifying the impact of their work can reignite motivation.

If you identify a motivation issue, consider interventions like:
- Offering stretch assignments that challenge them.
- Involving them in strategic discussions to give them a sense of ownership.
- Connecting them with senior leaders for mentorship.

Also, ask them directly about their learning goals. What skills do they wish to develop next? This not only shows you care but can also help them feel more engaged. Document your conversation and agreed a

---
## Phase 8 — Interactive demo UI (ipywidgets)

In [23]:
# ── Install required package (run once) ──────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'flask', '-q'])
print('✅ Flask ready')

✅ Flask ready


In [24]:
# ── Phase 8: Launch coaching assistant as a web app ──────────────────────────
import threading
import socket
import time
from flask import Flask, request, jsonify, Response
from IPython.display import display, HTML

def find_free_port(start=7860):
    for port in range(start, start + 20):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('127.0.0.1', port)) != 0:
                return port
    return start

PORT = find_free_port()

assert callable(chat_openai),    "❌ run Phase 1 first"
assert callable(retrieve),       "❌ run Phase 2 first"
assert callable(score_response), "❌ run Phase 4 first"
assert len(PROFILES) > 0,        "❌ run Phase 5 first"
print(f"✅ All dependencies OK — port {PORT}")

flask_app = Flask(__name__)

@flask_app.after_request
def add_cors(response):
    response.headers['Access-Control-Allow-Origin'] = '*'
    response.headers['Access-Control-Allow-Headers'] = 'Content-Type'
    response.headers['Access-Control-Allow-Methods'] = 'GET, POST, OPTIONS'
    return response

@flask_app.route('/chat', methods=['OPTIONS'])
def chat_preflight():
    return '', 204

STYLE_MAP = {
    'directive':     'Give clear, direct advice and concrete action steps.',
    'non-directive': 'Ask open questions to guide the user to their own answers. Avoid giving direct advice.',
    'balanced':      'Mix open questions with practical advice. Be warm but concrete.',
}

@flask_app.route('/chat', methods=['POST'])
def chat():
    try:
        data     = request.json
        message  = data['message']
        do_score = data.get('score', False)
        profile  = data['profile']
        history  = data.get('history', [])

        goals_str = ', '.join(profile.get('goals', []))
        system = f"""You are an expert coaching assistant for managers, built by Incenteev.
You are speaking with {profile['name']}, a {profile['role']} at {profile['company']}.
Their coaching goals are: {goals_str}.
Coaching style: {STYLE_MAP.get(profile['style'], '')}
Always be empathetic, specific, and practical. Keep responses under 180 words."""

        context_chunks = retrieve(message, k=3)
        system += "\n\n[Relevant coaching knowledge]\n" + "\n---\n".join(context_chunks)

        messages = [{'role': 'system', 'content': system}] + history
        response_text = chat_openai(messages)

        score = None
        if do_score:
            try:
                score = score_response(message, response_text).model_dump()
            except AttributeError:
                score = score_response(message, response_text).dict()

        return jsonify({'response': response_text, 'score': score})

    except Exception as e:
        import traceback
        return jsonify({'error': str(e), 'trace': traceback.format_exc()}), 500

@flask_app.route('/')
def index():
    return Response(HTML_APP, mimetype='text/html')

# ── Build HTML inline — no external file needed ───────────────────────────────
HTML_APP = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Coaching Assistant — Incenteev</title>
<link href="https://fonts.googleapis.com/css2?family=Syne:wght@400;500;600;700&family=Instrument+Sans:wght@300;400;500&display=swap" rel="stylesheet">
<style>
  :root {{
    --bg:#0a0a0f; --surface:#12121a; --surface2:#1a1a25;
    --border:#ffffff0f; --border2:#ffffff18;
    --accent:#7c6dff; --accent2:#a78bfa;
    --text:#e8e6f0; --text2:#8880a8; --text3:#44405a;
    --user-bg:#7c6dff; --coach-bg:#16161f; --green:#22c55e;
  }}
  * {{ box-sizing:border-box; margin:0; padding:0; }}
  body {{ font-family:'Instrument Sans',sans-serif; background:var(--bg); color:var(--text); height:100vh; display:flex; overflow:hidden; }}
  .sidebar {{ width:260px; background:var(--surface); border-right:1px solid var(--border); display:flex; flex-direction:column; flex-shrink:0; padding:20px 0; }}
  .sidebar-logo {{ display:flex; align-items:center; gap:10px; padding:0 18px 20px; border-bottom:1px solid var(--border); margin-bottom:20px; }}
  .logo-mark {{ width:36px; height:36px; background:linear-gradient(135deg,var(--accent),var(--accent2)); border-radius:10px; display:flex; align-items:center; justify-content:center; font-family:'Syne',sans-serif; font-weight:700; font-size:16px; color:white; flex-shrink:0; }}
  .logo-text {{ line-height:1.2; }}
  .logo-name {{ font-family:'Syne',sans-serif; font-size:14px; font-weight:600; color:var(--text); }}
  .logo-sub {{ font-size:10px; color:var(--text3); margin-top:1px; }}
  .sidebar-section {{ padding:0 14px; margin-bottom:24px; flex:1; overflow-y:auto; }}
  .sidebar-label {{ font-size:10px; font-weight:500; color:var(--text3); letter-spacing:.08em; text-transform:uppercase; padding:0 4px; margin-bottom:8px; }}
  .profile-card {{ background:var(--surface2); border:1px solid var(--border); border-radius:10px; padding:10px 12px; cursor:pointer; transition:border-color .15s,background .15s; margin-bottom:4px; display:flex; align-items:center; gap:10px; }}
  .profile-card:hover {{ border-color:var(--border2); background:#1e1e2e; }}
  .profile-card.active {{ border-color:var(--accent); background:#1e1a35; }}
  .profile-avatar {{ width:32px; height:32px; border-radius:50%; display:flex; align-items:center; justify-content:center; font-size:14px; flex-shrink:0; background:var(--surface); border:1px solid var(--border2); }}
  .profile-info {{ overflow:hidden; }}
  .profile-name {{ font-size:12px; font-weight:500; color:var(--text); white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }}
  .profile-role {{ font-size:10.5px; color:var(--text2); margin-top:1px; }}
  .style-badge {{ margin-left:auto; flex-shrink:0; font-size:9px; font-weight:500; padding:2px 6px; border-radius:20px; text-transform:uppercase; letter-spacing:.05em; }}
  .style-badge.directive {{ background:#ff6b6b15; color:#ff6b6b; border:1px solid #ff6b6b30; }}
  .style-badge.non-directive {{ background:#6bffb815; color:#6bffb8; border:1px solid #6bffb830; }}
  .style-badge.balanced {{ background:#ffcc6b15; color:#ffcc6b; border:1px solid #ffcc6b30; }}
  .status-bar {{ margin-top:auto; padding:14px 18px 0; border-top:1px solid var(--border); flex-shrink:0; }}
  .status-row {{ display:flex; align-items:center; gap:6px; margin-bottom:6px; }}
  .status-dot {{ width:6px; height:6px; border-radius:50%; background:var(--green); flex-shrink:0; animation:glow 2s infinite; }}
  @keyframes glow {{ 0%,100%{{opacity:1;}} 50%{{opacity:.5;}} }}
  .status-text {{ font-size:11px; color:var(--text2); }}
  .model-info {{ font-size:10px; color:var(--text3); }}
  .main {{ flex:1; display:flex; flex-direction:column; overflow:hidden; }}
  .chat-header {{ padding:16px 24px; border-bottom:1px solid var(--border); display:flex; align-items:center; justify-content:space-between; flex-shrink:0; background:var(--surface); }}
  .chat-title {{ font-family:'Syne',sans-serif; font-size:15px; font-weight:600; color:var(--text); }}
  .chat-sub {{ font-size:11px; color:var(--text2); margin-top:2px; }}
  .header-controls {{ display:flex; align-items:center; gap:12px; }}
  .toggle-wrap {{ display:flex; align-items:center; gap:6px; font-size:12px; color:var(--text2); cursor:pointer; }}
  .toggle-wrap input {{ accent-color:var(--accent); cursor:pointer; }}
  .clear-btn {{ background:transparent; border:1px solid var(--border2); color:var(--text2); padding:5px 12px; border-radius:7px; font-size:11px; font-family:'Instrument Sans',sans-serif; cursor:pointer; transition:all .15s; }}
  .clear-btn:hover {{ border-color:var(--accent); color:var(--accent); }}
  .messages {{ flex:1; overflow-y:auto; padding:28px 32px; display:flex; flex-direction:column; gap:20px; scroll-behavior:smooth; }}
  .messages::-webkit-scrollbar {{ width:3px; }}
  .messages::-webkit-scrollbar-thumb {{ background:var(--border2); border-radius:4px; }}
  .empty-state {{ display:flex; flex-direction:column; align-items:center; justify-content:center; flex:1; gap:16px; color:var(--text3); animation:fadeIn .4s ease; min-height:300px; }}
  @keyframes fadeIn {{ from{{opacity:0;transform:translateY(10px);}} to{{opacity:1;transform:none;}} }}
  .empty-icon {{ width:64px; height:64px; background:var(--surface2); border:1px solid var(--border2); border-radius:20px; display:flex; align-items:center; justify-content:center; font-size:28px; }}
  .empty-title {{ font-family:'Syne',sans-serif; font-size:20px; font-weight:600; color:#3a3650; text-align:center; }}
  .empty-body {{ font-size:13px; color:var(--text3); text-align:center; max-width:340px; line-height:1.7; }}
  .suggestions {{ display:flex; flex-wrap:wrap; gap:8px; justify-content:center; max-width:480px; margin-top:4px; }}
  .sug-btn {{ background:var(--surface2); border:1px solid var(--border); color:var(--text2); padding:8px 16px; border-radius:24px; font-size:12px; font-family:'Instrument Sans',sans-serif; cursor:pointer; transition:all .15s; }}
  .sug-btn:hover {{ border-color:var(--accent); color:var(--accent2); background:#1a1830; }}
  .msg {{ display:flex; gap:12px; animation:msgIn .2s ease; max-width:820px; }}
  @keyframes msgIn {{ from{{opacity:0;transform:translateY(6px);}} to{{opacity:1;transform:none;}} }}
  .msg.user {{ flex-direction:row-reverse; align-self:flex-end; }}
  .msg.assistant {{ align-self:flex-start; }}
  .av {{ width:34px; height:34px; border-radius:50%; display:flex; align-items:center; justify-content:center; font-size:15px; flex-shrink:0; margin-top:2px; }}
  .av.u {{ background:var(--user-bg); }}
  .av.a {{ background:var(--surface2); border:1px solid var(--border2); }}
  .bubble {{ padding:12px 16px; font-size:14px; line-height:1.7; max-width:640px; }}
  .msg.user .bubble {{ background:var(--user-bg); color:white; border-radius:18px 18px 4px 18px; }}
  .msg.assistant .bubble {{ background:var(--coach-bg); border:1px solid var(--border2); color:var(--text); border-radius:18px 18px 18px 4px; }}
  .score-block {{ margin-top:10px; padding:10px 14px; background:#0d0d16; border:1px solid var(--border); border-radius:10px; }}
  .score-row {{ display:flex; flex-wrap:wrap; gap:12px; margin-bottom:6px; }}
  .score-item {{ display:flex; align-items:center; gap:5px; }}
  .sc-label {{ font-size:11px; color:var(--text2); font-weight:500; }}
  .sc-stars {{ font-size:12px; color:#f5c518; letter-spacing:1px; }}
  .sc-fb {{ font-size:11px; color:var(--text2); font-style:italic; }}
  .thinking {{ display:flex; gap:5px; align-items:center; padding:6px 0; }}
  .dot {{ width:7px; height:7px; border-radius:50%; background:var(--accent); animation:blink 1.3s infinite; }}
  .dot:nth-child(2) {{ animation-delay:.2s; }} .dot:nth-child(3) {{ animation-delay:.4s; }}
  @keyframes blink {{ 0%,100%{{opacity:.25;transform:scale(.85);}} 50%{{opacity:1;transform:scale(1.1);}} }}
  .input-wrap {{ padding:16px 32px 20px; border-top:1px solid var(--border); background:var(--bg); flex-shrink:0; }}
  .input-box {{ background:var(--surface); border:1px solid var(--border2); border-radius:14px; display:flex; align-items:flex-end; gap:10px; padding:10px 14px; transition:border-color .2s; }}
  .input-box:focus-within {{ border-color:#7c6dff44; box-shadow:0 0 0 3px #7c6dff08; }}
  textarea {{ flex:1; background:transparent; border:none; outline:none; color:var(--text); font-family:'Instrument Sans',sans-serif; font-size:14px; line-height:1.6; resize:none; max-height:140px; min-height:24px; padding:2px 0; }}
  textarea::placeholder {{ color:var(--text3); }}
  .send-btn {{ width:38px; height:38px; background:var(--accent); border:none; border-radius:10px; cursor:pointer; flex-shrink:0; display:flex; align-items:center; justify-content:center; transition:background .15s,transform .1s; }}
  .send-btn:hover {{ background:#8c7dff; }} .send-btn:active {{ transform:scale(.92); }} .send-btn:disabled {{ background:var(--border2); cursor:not-allowed; }}
  .send-btn svg {{ width:17px; height:17px; fill:white; }}
  .input-footer {{ display:flex; justify-content:space-between; align-items:center; margin-top:8px; padding:0 2px; }}
  .hint {{ font-size:11px; color:var(--text3); }}
  .rag-badge {{ font-size:10px; color:var(--text3); display:flex; align-items:center; gap:5px; background:var(--surface2); border:1px solid var(--border); padding:3px 9px; border-radius:20px; }}
  .rag-dot {{ width:5px; height:5px; border-radius:50%; background:var(--green); }}
  .toast {{ position:fixed; bottom:24px; right:24px; background:#ff4444; color:white; padding:10px 18px; border-radius:10px; font-size:13px; z-index:200; display:none; }}
</style>
</head>
<body>
<div class="toast" id="toast"></div>
<div class="sidebar">
  <div class="sidebar-logo">
    <div class="logo-mark">C</div>
    <div class="logo-text"><div class="logo-name">CoachAI</div><div class="logo-sub">by Incenteev</div></div>
  </div>
  <div class="sidebar-section">
    <div class="sidebar-label">Select profile</div>
    <div id="profileList"></div>
  </div>
  <div class="status-bar">
    <div class="status-row"><div class="status-dot"></div><div class="status-text">GPT-4o-mini connected</div></div>
    <div class="model-info">RAG · FAISS · Scoring enabled</div>
  </div>
</div>
<div class="main">
  <div class="chat-header">
    <div><div class="chat-title" id="headerTitle">Coaching Assistant</div><div class="chat-sub" id="headerSub">Select a profile to begin</div></div>
    <div class="header-controls">
      <label class="toggle-wrap"><input type="checkbox" id="scoreToggle"> Show quality score</label>
      <button class="clear-btn" onclick="clearChat()">Clear chat</button>
    </div>
  </div>
  <div class="messages" id="messages">
    <div class="empty-state" id="emptyState">
      <div class="empty-icon">🎯</div>
      <div class="empty-title">Start a coaching conversation</div>
      <div class="empty-body">Select a profile on the left, then ask anything about managing your team. The AI adapts to each manager's coaching style and goals.</div>
      <div class="suggestions">
        <button class="sug-btn" onclick="fillInput('My team member keeps missing deadlines. What do I do?')">Missed deadlines</button>
        <button class="sug-btn" onclick="fillInput('How do I give feedback without demotivating?')">Giving feedback</button>
        <button class="sug-btn" onclick="fillInput('My best employee seems disengaged lately.')">Disengaged employee</button>
        <button class="sug-btn" onclick="fillInput('Two people on my team are in conflict.')">Team conflict</button>
        <button class="sug-btn" onclick="fillInput('How do I run a better 1-on-1 meeting?')">Better 1-on-1s</button>
        <button class="sug-btn" onclick="fillInput('One of my reports is not meeting expectations.')">Underperformance</button>
      </div>
    </div>
  </div>
  <div class="input-wrap">
    <div class="input-box">
      <textarea id="msgInput" placeholder="Ask a coaching question…" rows="1" onkeydown="handleKey(event)" oninput="autoResize(this)"></textarea>
      <button class="send-btn" id="sendBtn" onclick="sendMessage()">
        <svg viewBox="0 0 24 24"><path d="M2.01 21L23 12 2.01 3 2 10l15 2-15 2z"/></svg>
      </button>
    </div>
    <div class="input-footer">
      <span class="hint">Enter to send · Shift+Enter for new line</span>
      <span class="rag-badge"><span class="rag-dot"></span> RAG · gpt-4o-mini</span>
    </div>
  </div>
</div>
<script>
const PROFILES = [
  {{ id:'pg_001', name:'Alice Martin',       role:'Sales Manager',      company:'P&G',           style:'non-directive', goals:['Improve 1-on-1s','Give better feedback'],       emoji:'👩' }},
  {{ id:'or_001', name:'Mohammed Al-Rashid', role:'Engineering Lead',    company:'Orange',        style:'directive',     goals:['Build team autonomy','Reduce micromanagement'], emoji:'👨' }},
  {{ id:'sg_001', name:'Marie Fontaine',     role:'Risk Director',       company:'Soc. Generale', style:'directive',     goals:['Develop leaders','Reduce turnover'],             emoji:'👩' }},
  {{ id:'lg_001', name:'Ji-ho Kim',          role:'R&D Team Lead',       company:'LG',            style:'directive',     goals:['Encourage initiative','Reduce hierarchy'],       emoji:'👨' }},
  {{ id:'gr_002', name:'Isabelle Moreau',    role:'Claims Supervisor',   company:'Groupama',      style:'balanced',      goals:['Reduce team stress','Improve call quality'],     emoji:'👩' }},
  {{ id:'ot_007', name:'Priya Nair',         role:'Engineering Manager', company:'Capgemini',     style:'non-directive', goals:['Psychological safety','Reduce conflicts'],       emoji:'👩' }},
];

let activeProfile = PROFILES[0];
let sessions = {{}};
let showScore = false;

function init() {{
  renderProfiles();
  selectProfile(PROFILES[0]);
  document.getElementById('scoreToggle').addEventListener('change', e => {{ showScore = e.target.checked; }});
}}

function renderProfiles() {{
  document.getElementById('profileList').innerHTML = PROFILES.map(p => `
    <div class="profile-card" id="card-${{p.id}}" onclick="selectProfile(PROFILES.find(x=>x.id==='${{p.id}}'))">
      <div class="profile-avatar">${{p.emoji}}</div>
      <div class="profile-info">
        <div class="profile-name">${{p.name}}</div>
        <div class="profile-role">${{p.role}} · ${{p.company}}</div>
      </div>
      <span class="style-badge ${{p.style}}">${{p.style === 'non-directive' ? 'open' : p.style}}</span>
    </div>`).join('');
}}

function selectProfile(profile) {{
  activeProfile = profile;
  document.querySelectorAll('.profile-card').forEach(c => c.classList.remove('active'));
  document.getElementById('card-' + profile.id).classList.add('active');
  document.getElementById('headerTitle').textContent = profile.name;
  document.getElementById('headerSub').textContent = profile.role + ' · ' + profile.company + ' · ' + profile.style + ' style';
  renderMessages();
}}

async function sendMessage() {{
  const inp = document.getElementById('msgInput');
  const text = inp.value.trim();
  if (!text) return;
  inp.value = ''; inp.style.height = 'auto';
  document.getElementById('sendBtn').disabled = true;

  const pid = activeProfile.id;
  if (!sessions[pid]) sessions[pid] = [];
  sessions[pid].push({{ role: 'user', content: text }});
  renderMessages();
  showThinking();

  try {{
    const res = await fetch('http://127.0.0.1:{PORT}/chat', {{
      method: 'POST',
      headers: {{ 'Content-Type': 'application/json' }},
      body: JSON.stringify({{
        message: text,
        score: showScore,
        profile: activeProfile,
        history: sessions[pid].slice(-10)
      }})
    }});
    const data = await res.json();
    if (data.error) {{ showToast('Server error: ' + data.error); removeThinking(); return; }}
    sessions[pid].push({{ role: 'assistant', content: data.response }});
    removeThinking();
    renderMessages(data.score || null);
  }} catch(e) {{
    showToast('Could not reach server — is the notebook still running? ' + e.message);
    removeThinking();
  }} finally {{
    document.getElementById('sendBtn').disabled = false;
  }}
}}

function renderMessages(latestScore = null) {{
  const area = document.getElementById('messages');
  const pid = activeProfile.id;
  const history = sessions[pid] || [];
  if (history.length === 0) {{
    area.innerHTML = `
      <div class="empty-state" id="emptyState">
        <div class="empty-icon">🎯</div>
        <div class="empty-title">Start a coaching conversation</div>
        <div class="empty-body">Ask anything about managing your team. The AI adapts to ${{activeProfile.name}}'s ${{activeProfile.style}} coaching style.</div>
        <div class="suggestions">
          <button class="sug-btn" onclick="fillInput('My team member keeps missing deadlines.')">Missed deadlines</button>
          <button class="sug-btn" onclick="fillInput('How do I give feedback without demotivating?')">Giving feedback</button>
          <button class="sug-btn" onclick="fillInput('My best employee seems disengaged lately.')">Disengaged employee</button>
          <button class="sug-btn" onclick="fillInput('Two people on my team are in conflict.')">Team conflict</button>
          <button class="sug-btn" onclick="fillInput('How do I run a better 1-on-1 meeting?')">Better 1-on-1s</button>
          <button class="sug-btn" onclick="fillInput('One of my reports is not meeting expectations.')">Underperformance</button>
        </div>
      </div>`;
    return;
  }}
  area.innerHTML = history.map((m, i) => {{
    const isUser = m.role === 'user';
    const isLast = i === history.length - 1;
    const scoreHTML = (!isUser && isLast && latestScore) ? buildScoreHTML(latestScore) : '';
    return `
      <div class="msg ${{m.role}}">
        <div class="av ${{isUser ? 'u' : 'a'}}">${{isUser ? '👤' : '🎯'}}</div>
        <div class="bubble">${{m.content.replace(/\\n/g,'<br>')}}${{scoreHTML}}</div>
      </div>`;
  }}).join('');
  area.scrollTop = area.scrollHeight;
}}

function buildScoreHTML(score) {{
  const dims = ['relevance','specificity','actionability','empathy','overall'];
  const stars = dims.map(d =>
    `<span class="score-item"><span class="sc-label">${{d}}</span><span class="sc-stars">${{'★'.repeat(score[d] || 0)}}${{' ☆'.repeat(5 - (score[d] || 0))}}</span></span>`
  ).join('');
  return `<div class="score-block"><div class="score-row">${{stars}}</div><div class="sc-fb">💬 ${{score.feedback || ''}}</div></div>`;
}}

function showThinking() {{
  const area = document.getElementById('messages');
  const es = document.getElementById('emptyState'); if (es) es.remove();
  const row = document.createElement('div');
  row.className = 'msg assistant'; row.id = 'thinkRow';
  row.innerHTML = `<div class="av a">🎯</div><div class="bubble"><div class="thinking"><div class="dot"></div><div class="dot"></div><div class="dot"></div></div></div>`;
  area.appendChild(row);
  area.scrollTop = area.scrollHeight;
}}

function removeThinking() {{ const t = document.getElementById('thinkRow'); if (t) t.remove(); }}
function clearChat() {{ sessions[activeProfile.id] = []; renderMessages(); }}
function fillInput(text) {{ const inp = document.getElementById('msgInput'); inp.value = text; autoResize(inp); inp.focus(); }}
function autoResize(el) {{ el.style.height = 'auto'; el.style.height = Math.min(el.scrollHeight, 140) + 'px'; }}
function handleKey(e) {{ if (e.key === 'Enter' && !e.shiftKey) {{ e.preventDefault(); sendMessage(); }} }}
function showToast(msg) {{ const t = document.getElementById('toast'); t.textContent = msg; t.style.display = 'block'; setTimeout(() => {{ t.style.display = 'none'; }}, 4000); }}

init();
</script>
</body></html>"""

def run_flask():
    flask_app.run(host='0.0.0.0', port=PORT, debug=False, use_reloader=False)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()
time.sleep(1.5)

url = f'http://127.0.0.1:{PORT}'
print(f'✅ Coaching Assistant live at {url}')

display(HTML(f'''
<div style="font-family:sans-serif;background:#12121a;border:1px solid #2a2a38;border-radius:14px;padding:24px 28px;max-width:460px;margin:12px 0;">
  <div style="display:flex;align-items:center;gap:10px;margin-bottom:14px">
    <div style="width:10px;height:10px;border-radius:50%;background:#22c55e"></div>
    <span style="color:#22c55e;font-size:13px;font-weight:600">Server running on port {PORT}</span>
  </div>
  <a href="{url}" target="_blank" style="display:inline-flex;align-items:center;gap:8px;background:#7c6dff;color:white;padding:12px 22px;border-radius:10px;text-decoration:none;font-size:14px;font-weight:600;">
    🚀 Open Coaching Assistant
  </a>
  <div style="margin-top:12px;font-size:11px;color:#44405a">Keep this notebook running while using the app.</div>
</div>
'''))

✅ All dependencies OK — port 7861
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:7861
 * Running on http://192.168.1.72:7861
Press CTRL+C to quit


✅ Coaching Assistant live at http://127.0.0.1:7861


127.0.0.1 - - [25/Apr/2026 19:10:01] "GET / HTTP/1.1" 200 -


---
## 📋 Quick reference

| What | Where |
|------|-------|
| Change LLM | `ACTIVE_MODEL` in Phase 1 |
| Add documents | Append to `COACHING_DOCS` in Phase 2, then re-run `build_faiss_index()` |
| Add user profile | `get_or_create_profile(...)` in Phase 5 |
| API docs | http://127.0.0.1:8000/docs |
| Test API | Phase 7 — httpx test cell |
| Interactive UI | Phase 8 — ipywidgets chat |

### Next steps
- Swap `COACHING_DOCS` for real PDFs using `langchain_community.document_loaders.PyPDFLoader`
- Persist FAISS index: `faiss.write_index(index, "coaching.index")` / `faiss.read_index(...)`
- Persist profiles: replace `dict` store with SQLite using `sqlite3` or SQLAlchemy
- Add LangGraph for multi-step coaching workflows (diagnosis → advice → follow-up)
- Add pytest: test retrieval relevance and scorer consistency